# MMHCL+ Training for Amazon Baby Dataset (Local)
## MMHCL via Neighbor-Layer Hypergraph Contrastive Learning

This notebook trains the **MMHCL+** model on the Amazon Baby dataset on a local machine with NVIDIA RTX 5090 (32GB VRAM).

MMHCL+ extends the original MMHCL (Multi-Modal Hypergraph Contrastive Learning) architecture with:
- **Neighbor-Layer CL** — positive pairs from adjacent GNN layers (NLGCL+ approach)
- **Barlow Twins** on the user-user hypergraph branch (u2u)
- **Chunked InfoNCE** on the item-item hypergraph branch (i2i) with Adaptive Sample Weighting
- **Soft BYOL** cross-view alignment (hypergraph → bipartite embeddings)
- **Dirichlet Energy** regularisation to prevent over-smoothing
- **GradNorm** dynamic loss balancing across 5 tasks
- **Warmup epochs** + exponential temperature annealing

## Configuration:
- **GPU**: NVIDIA RTX 5090 (32GB VRAM)
- **Batch Size**: 1024 (fit with original paper)
- **Max Epochs**: 250 (fit with original paper)
- **Warmup Epochs**: 10 (BPR-only phase to stabilise base embeddings)
- **W&B Project**: snips-local-mmhcl-plus-baby
- **Training Script**: `codes/main_mmhcl_plus.py`

## Steps:
1. Verify environment and GPU
2. Setup W&B logging
3. Train MMHCL+ model with multiple seeds
4. Export results to Excel

## 1. Environment Setup & GPU Verification

In [2]:
from __future__ import annotations

import os
import sys
import torch

# Get the project root directory (handle case where we might already be in codes dir)
current_dir: str = os.getcwd()
if current_dir.endswith('codes'):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir

CODES_DIR: str = os.path.join(PROJECT_ROOT, 'codes')
DATA_DIR: str = os.path.join(PROJECT_ROOT, 'data')

print(f"Project Root: {PROJECT_ROOT}")
print(f"Codes Directory: {CODES_DIR}")
print(f"Data Directory: {DATA_DIR}")

# Verify directories exist
assert os.path.exists(CODES_DIR), f"Codes directory not found: {CODES_DIR}"
assert os.path.exists(DATA_DIR), f"Data directory not found: {DATA_DIR}"
print("\n[OK] All directories verified")

# Verify GPU
print("\n" + "="*60)
print("GPU Information")
print("="*60)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected! Training will be very slow.")

Project Root: c:\Users\Anh Khoi\Desktop\MyCode\Experiments\5090_Professional_Original_paper_MMHCL_randoms_Amazon_Baby
Codes Directory: c:\Users\Anh Khoi\Desktop\MyCode\Experiments\5090_Professional_Original_paper_MMHCL_randoms_Amazon_Baby\codes
Data Directory: c:\Users\Anh Khoi\Desktop\MyCode\Experiments\5090_Professional_Original_paper_MMHCL_randoms_Amazon_Baby\data

[OK] All directories verified

GPU Information
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
CUDA version: 12.8
GPU Memory: 31.8 GB


## 1.5 Install Dependencies (Run Once)

In [3]:
# Install required dependencies (run this cell once)
# Configuration: PyTorch 2.0.1+cu118, NumPy 1.24.3, SciPy 1.10.1, scikit-learn 1.2.2
from __future__ import annotations

import subprocess
import sys
import os

print("="*60)
print("Installing Dependencies")
print("  - PyTorch 2.0.1+cu118")
print("  - NumPy 1.24.3")
print("  - SciPy 1.10.1")
print("  - scikit-learn 1.2.2")
print("="*60)

# Check if UV-managed Python (needs --break-system-packages)
pip_extra_args: list[str] = []
test_result: subprocess.CompletedProcess[str] = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--dry-run', 'pip'],
    capture_output=True, text=True,
)
if 'externally-managed' in test_result.stderr:
    pip_extra_args = ['--break-system-packages']
    print("\n[INFO] UV-managed Python detected, using --break-system-packages")

# Step 1: Install from requirements.txt
requirements_path: str = os.path.join(PROJECT_ROOT, 'requirements.txt')
if os.path.exists(requirements_path):
    print(f"\n1. Installing packages from: {requirements_path}")
    print("   This may take several minutes for PyTorch (~2.6GB)...")
    cmd: list[str] = [sys.executable, '-m', 'pip', 'install', '-r', requirements_path] + pip_extra_args
    result: subprocess.CompletedProcess[str] = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"   [ERROR] Installation failed! Exit code: {result.returncode}")
        print("\n--- STDERR ---")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
        raise Exception("Failed to install requirements")
    print("   [OK] requirements.txt packages installed")
else:
    print(f"   [WARNING] requirements.txt not found at {requirements_path}")

# Step 2: Install additional packages
additional_packages: list[str] = ['wandb', 'openpyxl', 'pandas']
print(f"\n2. Installing additional packages: {additional_packages}")
for package in additional_packages:
    print(f"   Installing {package}...")
    cmd: list[str] = [sys.executable, '-m', 'pip', 'install', '-q', package] + pip_extra_args
    subprocess.run(cmd, capture_output=True)
print("   [OK] Additional packages installed")

print("\n" + "="*60)
print("[OK] All packages installed successfully!")
print("="*60)
print("\n[IMPORTANT] Please RESTART THE KERNEL before running the next cells!")

Installing Dependencies
  - PyTorch 2.0.1+cu118
  - NumPy 1.24.3
  - SciPy 1.10.1
  - scikit-learn 1.2.2

1. Installing packages from: c:\Users\Anh Khoi\Desktop\MyCode\Experiments\5090_Professional_Original_paper_MMHCL_randoms_Amazon_Baby\requirements.txt
   This may take several minutes for PyTorch (~2.6GB)...
   [OK] requirements.txt packages installed

2. Installing additional packages: ['wandb', 'openpyxl', 'pandas']
   Installing wandb...
   Installing openpyxl...
   Installing pandas...
   [OK] Additional packages installed

[OK] All packages installed successfully!

[IMPORTANT] Please RESTART THE KERNEL before running the next cells!


## 2. Weights & Biases Setup

In [1]:
from __future__ import annotations

import wandb
from IPython.display import display, Markdown

# ========== Login to Weights & Biases ==========
# W&B Configuration
wandb_project: str = 'snips-local-mmhcl-baby'
wandb_entity: str = 'baitapck51cc-uet'

# Your W&B API key
WANDB_API_KEY: str = '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'

# Login to W&B
wandb.login(key=WANDB_API_KEY)

# Verify login
print("✓ W&B login successful")
print(f"✓ Entity: {wandb_entity}")
print(f"✓ Project: {wandb_project}")

# Construct W&B URL and display clickable link
wandb_url: str = f"https://wandb.ai/{wandb_entity}/{wandb_project}"
print()
display(Markdown(f"## 📊 [Click here to open Weights & Biases Dashboard]({wandb_url})"))
print(f"Direct Link: {wandb_url}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Anh Khoi\_netrc
wandb: Currently logged in as: baitapck51cc (baitapck51cc-uet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ W&B login successful
✓ Entity: baitapck51cc-uet
✓ Project: snips-local-mmhcl-baby



## 📊 [Click here to open Weights & Biases Dashboard](https://wandb.ai/baitapck51cc-uet/snips-local-mmhcl-baby)

Direct Link: https://wandb.ai/baitapck51cc-uet/snips-local-mmhcl-baby


## 2.5 GPU Memory Monitor (Optional)

Start a lightweight background monitor to print GPU VRAM usage while training.
Run the stop function after training completes.

In [2]:
from __future__ import annotations

import threading
import time
import torch
import csv
import os
from typing import Optional

# GPU monitor controls
GPU_MONITOR_RUNNING: bool = False
GPU_MONITOR_THREAD: Optional[threading.Thread] = None


def _resolve_monitor_root() -> str:
    current_dir: str = os.getcwd()
    if current_dir.endswith('codes'):
        return os.path.dirname(current_dir)
    return current_dir


def _gpu_monitor(interval_seconds: int, csv_path: str, print_live: bool) -> None:
    global GPU_MONITOR_RUNNING
    if not torch.cuda.is_available():
        print("[WARNING] CUDA not available. GPU monitor will not run.")
        GPU_MONITOR_RUNNING = False
        return

    total_mem: int = torch.cuda.get_device_properties(0).total_memory
    start_time: float = time.time()

    file_exists: bool = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as csv_file:
        writer: csv.writer = csv.writer(csv_file)
        if not file_exists:
            writer.writerow([
                'relative_time_s',
                'allocated_gb',
                'reserved_gb',
                'allocated_pct',
                'reserved_pct'
            ])

        print(f"[INFO] GPU monitor started. Logging to: {csv_path}")
        while GPU_MONITOR_RUNNING:
            allocated: int = torch.cuda.memory_allocated(0)
            reserved: int = torch.cuda.memory_reserved(0)
            allocated_pct: float = (allocated / total_mem) * 100
            reserved_pct: float = (reserved / total_mem) * 100
            rel_time: float = time.time() - start_time

            writer.writerow([
                f"{rel_time:.2f}",
                f"{allocated / 1024**3:.4f}",
                f"{reserved / 1024**3:.4f}",
                f"{allocated_pct:.2f}",
                f"{reserved_pct:.2f}",
            ])
            csv_file.flush()

            if print_live:
                print(
                    f"[GPU] allocated={allocated/1024**3:.2f} GB ({allocated_pct:.1f}%), "
                    f"reserved={reserved/1024**3:.2f} GB ({reserved_pct:.1f}%)"
                )

            time.sleep(interval_seconds)

    print("[INFO] GPU monitor stopped.")


def start_gpu_monitor(interval_seconds: int = 10, csv_filename: str = 'gpu_memory_monitor.csv', print_live: bool = False) -> None:
    """Start background GPU monitor. Logs to CSV every N seconds."""
    global GPU_MONITOR_RUNNING, GPU_MONITOR_THREAD
    if GPU_MONITOR_RUNNING:
        print("[INFO] GPU monitor already running.")
        return

    monitor_root: str = _resolve_monitor_root()
    csv_path: str = os.path.join(monitor_root, csv_filename)

    GPU_MONITOR_RUNNING = True
    GPU_MONITOR_THREAD = threading.Thread(
        target=_gpu_monitor,
        args=(interval_seconds, csv_path, print_live),
        daemon=True
    )
    GPU_MONITOR_THREAD.start()


def stop_gpu_monitor() -> None:
    """Stop background GPU monitor."""
    global GPU_MONITOR_RUNNING
    GPU_MONITOR_RUNNING = False


# Start monitor with 15s interval (adjust as needed)
# Set print_live=True if you want console output too
start_gpu_monitor(interval_seconds=15, csv_filename='gpu_memory_monitor.csv', print_live=False)

# Call stop_gpu_monitor() after training completes.

[INFO] GPU monitor started. Logging to: c:\Users\Anh Khoi\Desktop\MyCode\Experiments\5090_Professional_Original_paper_MMHCL_randoms_Amazon_Baby\gpu_memory_monitor.csv


## 3. Multi-Seed Training (MMHCL+)

Train the **MMHCL+ (Neighbor-Layer Hypergraph CL)** model with multiple random seeds.

### Base Hyperparameters (same as original MMHCL):
- **Batch Size**: 1024 (fit with original paper, NVIDIA RTX 5090 32GB)
- **Max Epochs**: 250 (fit with original paper)
- **Learning Rate**: 0.0001
- **Early Stopping**: patience=30, min_epochs=75
- **ReduceLROnPlateau**: enabled (factor=0.5, patience=3)

### MMHCL+ Additions:
- **Warmup Epochs**: 10 — BPR-only phase before enabling CL losses
- **Temperature Schedule**: tau anneals from 0.5 → 0.05 with gamma=0.99
- **Barlow Twins** lambda=0.005 (u2u branch, expanded projector 64→256→128)
- **Chunked InfoNCE** chunk_size=1024 (i2i branch)
- **Soft BYOL** alignment weight=1.0 (hypergraph → bipartite stop-grad target)
- **Dirichlet Energy** weight=0.1 (over-smoothing regularisation)
- **GradNorm** alpha=1.5 (dynamic 5-task loss balancing)
- **WEMAManager**: initialised from image_feat.npy + text_feat.npy for item-side ASW

In [3]:
from __future__ import annotations

import os
import sys
import subprocess
import json
import re
import numpy as np
import time
import random
from pathlib import Path
from typing import Any, Optional

# Define project directories (handle case where we might already be in codes dir)
current_dir: str = os.getcwd()
if current_dir.endswith('codes'):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir
    
CODES_DIR: str = os.path.join(PROJECT_ROOT, 'codes')
DATA_DIR: str = os.path.join(PROJECT_ROOT, 'data')

# Change to codes directory (only if not already there)
if not os.getcwd().endswith('codes'):
    os.chdir(CODES_DIR)
if CODES_DIR not in sys.path:
    sys.path.insert(0, CODES_DIR)

# Get the Python executable path (same as Jupyter kernel)
PYTHON_EXE: str = sys.executable
    
print(f"Project Root: {PROJECT_ROOT}")
print(f"Working directory: {os.getcwd()}")
print(f"Python executable: {PYTHON_EXE}")

print("\n" + "="*80)
print("MMHCL+ MULTI-SEED TRAINING: Running 3 experiments with different random seeds")
print("="*80)

# Generate truly random seeds based on current time
base_seed: int = int(time.time_ns() % (2**31))
random.seed(base_seed)

# Generate 3 different random seeds
n_runs: int = 3
seeds: list[int] = [random.randint(1, 2**31 - 1) for _ in range(n_runs)]

print(f"\nGenerated {n_runs} random seeds based on current time:")
print(f"Base timestamp seed: {base_seed}")
print(f"Seeds for experiments: {seeds}")
print(f"(These seeds will be saved in the summary file for reproducibility)")

# W&B Configuration
wandb_project: str = 'snips-local-mmhcl-plus-baby'
wandb_entity: str = 'baitapck51cc-uet'

# Base hyperparameters (same as original paper)
dataset: str = 'Baby'
gpu_id: int = 0
epoch: int = 250  # Maximum epochs (fit with original paper)
verbose: int = 5

# Batch size 1024 (fit with original paper, RTX 5090 32GB VRAM)
batch_size: int = 1024
lr: float = 0.0001
regs: float = 1e-3
embed_size: int = 64
topk: int = 5
core: int = 5
User_layers: int = 3
Item_layers: int = 2
user_loss_ratio: float = 0.03
item_loss_ratio: float = 0.07
temperature: float = 0.6  # kept for path naming; actual tau uses tau_max/tau_min/tau_gamma

# ── MMHCL+ additions ──────────────────────────────────────────────────────────

# Warmup + temperature annealing (TEX §4.3)
warmup_epochs: int = 10          # BPR-only stabilisation before CL kicks in
tau_max: float = 0.5             # initial temperature for i2i InfoNCE
tau_min: float = 0.05            # floor temperature
tau_gamma: float = 0.99          # per-epoch decay factor

# Loss function settings
barlow_lambda: float = 5e-3      # off-diagonal penalty for Barlow Twins (u2u)
info_nce_chunk_size: int = 1024  # memory-safe InfoNCE chunk size (i2i)
gradnorm_alpha: float = 1.5      # GradNorm alpha for 5-task balancing

# Per-task initial weights (GradNorm adjusts dynamically)
bpr_weight: float = 1.0
u2u_weight: float = 1.0
i2i_weight: float = 1.0
align_weight: float = 1.0
dirichlet_weight: float = 0.1

# Expanded projector dimensions (u2u branch)
projector_hidden_dim: int = 256
projector_out_dim: int = 128

# WEMAManager (Adaptive Sample Weighting — item AND user sides)
w_ema_alpha: float = 0.9
w_ema_update_interval: int = 5

# Soft Topology-Aware Purification (TEX §3.4)
soft_purification_percentile: float = 0.8    # percentile threshold for continuous relaxation
soft_purification_temp: float = 0.2          # softmax temperature for topology weights

# EMA Teacher Network (TEX §4.4 Alg.1 Step 11: xi <- beta*xi + (1-beta)*theta)
ema_momentum: float = 0.99                   # teacher momentum (BYOL default)

# Profiling-Guided Activation Checkpointing (TEX §4.3 Table 2)
checkpoint_threshold: int = 1                # checkpoint layers >= this idx; -1 = disable

# ─────────────────────────────────────────────────────────────────────────────

# Early stopping configuration
early_stopping_patience: int = 30
early_stopping_min_epochs: int = 75  # Adjusted proportionally for 250 max epochs
early_stopping_min_delta: float = 0.0001
early_stopping_monitor: str = 'val_recall@20'
early_stopping_mode: str = 'max'
early_stopping_restore_best: int = 1
adaptive_patience: int = 0  # Disabled

# ReduceLROnPlateau scheduler
use_reduce_lr: int = 1
reduce_lr_factor: float = 0.5
reduce_lr_patience: int = 3
reduce_lr_min: float = 1e-6

# Storage for results
all_results: list[dict[str, Any]] = []

print(f"\n{'='*80}")
print("Training Configuration (MMHCL+):")
print(f"{'='*80}")
print(f"  Dataset:          {dataset}")
print(f"  GPU ID:           {gpu_id}")
print(f"  Max Epochs:       {epoch}")
print(f"  Batch Size:       {batch_size}")
print(f"  Learning Rate:    {lr}")
print(f"  Warmup Epochs:    {warmup_epochs}")
print(f"  Tau schedule:     {tau_max} -> {tau_min} (gamma={tau_gamma})")
print(f"  Early Stopping:   patience={early_stopping_patience}, min_epochs={early_stopping_min_epochs}")
print(f"  W&B Project:      {wandb_project}")
print(f"  Training Script:  main_mmhcl_plus.py")

# Run training for each seed
for run_idx, seed in enumerate(seeds, 1):
    print(f"\n{'='*80}")
    print(f"RUN {run_idx}/{n_runs}: Training MMHCL+ with seed={seed}")
    print(f"{'='*80}")

    # Build training command (use PYTHON_EXE to ensure same Python as Jupyter kernel)
    cmd: list[str] = [
        PYTHON_EXE, 'main_mmhcl_plus.py',
        # ── Base MMHCL args ──────────────────────────────────────────────────
        '--dataset', dataset,
        '--gpu_id', str(gpu_id),
        '--seed', str(seed),
        '--epoch', str(epoch),
        '--verbose', str(verbose),
        '--batch_size', str(batch_size),
        '--lr', str(lr),
        '--regs', str(regs),
        '--embed_size', str(embed_size),
        '--topk', str(topk),
        '--core', str(core),
        '--User_layers', str(User_layers),
        '--Item_layers', str(Item_layers),
        '--user_loss_ratio', str(user_loss_ratio),
        '--item_loss_ratio', str(item_loss_ratio),
        '--temperature', str(temperature),
        '--early_stopping_patience', str(early_stopping_patience),
        '--early_stopping_min_epochs', str(early_stopping_min_epochs),
        '--early_stopping_min_delta', str(early_stopping_min_delta),
        '--early_stopping_monitor', str(early_stopping_monitor),
        '--early_stopping_mode', str(early_stopping_mode),
        '--early_stopping_restore_best', str(early_stopping_restore_best),
        '--adaptive_patience', str(adaptive_patience),
        '--use_reduce_lr', str(use_reduce_lr),
        '--reduce_lr_factor', str(reduce_lr_factor),
        '--reduce_lr_patience', str(reduce_lr_patience),
        '--reduce_lr_min', str(reduce_lr_min),
        # ── MMHCL+ additions ─────────────────────────────────────────────────
        '--warmup_epochs', str(warmup_epochs),
        '--tau_max', str(tau_max),
        '--tau_min', str(tau_min),
        '--tau_gamma', str(tau_gamma),
        '--barlow_lambda', str(barlow_lambda),
        '--info_nce_chunk_size', str(info_nce_chunk_size),
        '--gradnorm_alpha', str(gradnorm_alpha),
        '--bpr_weight', str(bpr_weight),
        '--u2u_weight', str(u2u_weight),
        '--i2i_weight', str(i2i_weight),
        '--align_weight', str(align_weight),
        '--dirichlet_weight', str(dirichlet_weight),
        '--projector_hidden_dim', str(projector_hidden_dim),
        '--projector_out_dim', str(projector_out_dim),
        '--w_ema_alpha', str(w_ema_alpha),
        '--w_ema_update_interval', str(w_ema_update_interval),
        '--soft_purification_percentile', str(soft_purification_percentile),
        '--soft_purification_temp', str(soft_purification_temp),
        '--ema_momentum', str(ema_momentum),
        '--checkpoint_threshold', str(checkpoint_threshold),
        # ── W&B ──────────────────────────────────────────────────────────────
        '--use_wandb', '1',
        '--wandb_project', wandb_project,
        '--wandb_entity', wandb_entity,
        '--wandb_run_name', f'mmhcl_plus_seed_{seed}'
    ]

    print(f"Command: {' '.join(cmd)}")
    print(f"Current directory: {os.getcwd()}\n")

    # Run training with combined stdout/stderr for real-time output
    # Use PYTHONIOENCODING to avoid Windows encoding issues
    env: dict[str, str] = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    env['PYTHONUNBUFFERED'] = '1'

    result: subprocess.CompletedProcess[str] = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        env=env,
        encoding='utf-8',
        errors='replace'
    )

    # Always print stdout if there's any output
    if result.stdout:
        print(result.stdout)

    if result.returncode != 0:
        print(f"\n[WARNING] Training with seed={seed} exited with code {result.returncode}")
        # Show error output to help debug
        if result.stderr:
            print(f"\n[ERROR OUTPUT]:")
            print(result.stderr)
    else:
        print(f"\n[OK] Training with seed={seed} completed successfully")

    # Extract results from log file
    path_name: str = f"uu_ii={User_layers}_{Item_layers}_{user_loss_ratio}_{item_loss_ratio}_topk={topk}_t={temperature}_regs={regs}_dim={embed_size}_seed={seed}_"
    log_file: str = f"../{dataset}/{path_name}/{path_name}.txt"

    if os.path.exists(log_file):
        with open(log_file, 'r') as f:
            log_content: str = f.read()

        # Extract BEST test metrics
        recall_match: Optional[re.Match[str]] = re.search(r'BEST_Test_Recall@20:\s+([\d.]+)', log_content)
        precision_match: Optional[re.Match[str]] = re.search(r'BEST_Test_Precision@20:\s+([\d.]+)', log_content)
        ndcg_match: Optional[re.Match[str]] = re.search(r'BEST_Test_NDCG@20:\s+([\d.]+)', log_content)

        # Fallback to old format
        if not recall_match:
            recall_match = re.search(r'Test_Recall@20:\s+([\d.]+)', log_content)
        if not precision_match:
            precision_match = re.search(r'Test_Precision@20:\s+([\d.]+)', log_content)
        if not ndcg_match:
            ndcg_match = re.search(r'Test_NDCG@20:\s+([\d.]+)', log_content)

        if recall_match and precision_match and ndcg_match:
            run_result: dict[str, Any] = {
                'seed': seed,
                'recall@20': float(recall_match.group(1)),
                'precision@20': float(precision_match.group(1)),
                'ndcg@20': float(ndcg_match.group(1)),
                'log_file': log_file
            }
            all_results.append(run_result)
            print(f"  Extracted metrics: Recall@20={run_result['recall@20']:.6f}, "
                  f"Precision@20={run_result['precision@20']:.6f}, "
                  f"NDCG@20={run_result['ndcg@20']:.6f}")
        else:
            print(f"  Could not extract metrics from log file: {log_file}")
    else:
        print(f"  Log file not found: {log_file}")

print(f"\n{'='*80}")
print("ALL TRAINING RUNS COMPLETED")
print(f"{'='*80}")


Project Root: c:\Users\Anh Khoi\Desktop\MyCode\Experiments\5090_Professional_Original_paper_MMHCL_randoms_Amazon_Baby
Working directory: c:\Users\Anh Khoi\Desktop\MyCode\Experiments\5090_Professional_Original_paper_MMHCL_randoms_Amazon_Baby\codes
Python executable: c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe

MULTI-SEED TRAINING: Running 5 experiments with different random seeds

Generated 5 random seeds based on current time:
Base timestamp seed: 1204046920
Seeds for experiments: [1543344208, 5394289, 1327538177, 660375272, 1942296275]
(These seeds will be saved in the summary file for reproducibility)

Training Configuration:
  Dataset: Baby
  GPU ID: 0
  Max Epochs: 250
  Batch Size: 1024
  Learning Rate: 0.0001
  Early Stopping: patience=30, min_epochs=75
  W&B Project: snips-local-mmhcl-baby

RUN 1/5: Training with seed=1543344208
Command: c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe main.py --dataset Baby --gpu_id 0 --seed 1543344208 --epoch 250 --verbose 5 --batch

## 4. Results Summary & Statistics

In [4]:
from __future__ import annotations

from typing import Any
import numpy as np
import json

# Compute and display statistics
if len(all_results) > 0:
    print(f"\n{'='*80}")
    print(f"RESULTS SUMMARY (Mean ± Std across {len(all_results)} runs)")
    print(f"{'='*80}")

    metrics: list[str] = ['recall@20', 'precision@20', 'ndcg@20']
    stats: dict[str, dict[str, Any]] = {}

    for metric in metrics:
        values: list[float] = [r[metric] for r in all_results]
        mean_val: np.floating = np.mean(values)
        std_val: np.floating = np.std(values)
        stats[metric] = {'mean': mean_val, 'std': std_val, 'values': values}

        print(f"\n{metric.upper()}:")
        print(f"  Mean: {mean_val:.6f}")
        print(f"  Std:  {std_val:.6f}")
        print(f"  Mean ± Std: {mean_val:.6f} ± {std_val:.6f}")
        print(f"  Individual values: {[f'{v:.6f}' for v in values]}")

    # Compare with reference values (MMHCL paper has no Amazon Baby results).
    print(f"\n{'='*80}")
    print("COMPARISON WITH REFERENCE BASELINE (BM3 - Amazon Baby)")
    print("NOTE: MMHCL paper does not report Amazon Baby results.")
    print("      Reference: BM3 (WWW '23) Table 3 - per-user 8:1:1 split on Baby.")
    print(f"{'='*80}")
    paper_values: dict[str, float] = {
        'recall@20': 0.0883,
        'precision@20': 0.0044,
        'ndcg@20': 0.0383
    }
    for metric in metrics:
        our_mean: Any = stats[metric]['mean']
        paper_val: float = paper_values[metric]
        diff_pct: float = ((our_mean - paper_val) / paper_val) * 100
        status: str = "✓ OK" if abs(diff_pct) < 10 else "⚠️ CHECK"
        print(f"  {metric.upper()}: Ours={our_mean:.4f} vs Paper={paper_val:.4f} ({diff_pct:+.1f}%) [{status}]")

    # Save summary to file
    summary_file: str = f"../{dataset}/multi_seed_summary.json"
    summary_data: dict[str, Any] = {
        'n_runs': len(all_results),
        'seeds': [r['seed'] for r in all_results],
        'config': {
            'batch_size': batch_size,
            'max_epochs': epoch,
            'lr': lr,
            'early_stopping_patience': early_stopping_patience,
            'early_stopping_min_epochs': early_stopping_min_epochs,
            'wandb_project': wandb_project
        },
        'statistics': {k: {'mean': float(v['mean']), 'std': float(v['std'])} for k, v in stats.items()},
        'individual_results': all_results
    }

    with open(summary_file, 'w') as f:
        json.dump(summary_data, f, indent=2)

    print(f"\n✓ Summary saved to: {summary_file}")

    # Human-readable summary
    summary_txt_file: str = f"../{dataset}/multi_seed_summary.txt"
    with open(summary_txt_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write(f"MULTI-SEED TRAINING RESULTS ({len(all_results)} runs)\n")
        f.write(f"Configuration: batch_size={batch_size}, max_epochs={epoch}, lr={lr}\n")
        f.write("="*80 + "\n\n")
        f.write(f"Seeds used: {seeds[:len(all_results)]}\n\n")

        for metric in metrics:
            f.write(f"{metric.upper()}:\n")
            f.write(f"  Mean ± Std: {stats[metric]['mean']:.6f} ± {stats[metric]['std']:.6f}\n")
            f.write(f"  Individual: {[f'{v:.6f}' for v in stats[metric]['values']]}\n\n")

        f.write("\nIndividual Run Results:\n")
        f.write("-"*80 + "\n")
        for r in all_results:
            f.write(f"Seed {r['seed']:10d}: Recall@20={r['recall@20']:.6f}, "
                   f"Precision@20={r['precision@20']:.6f}, "
                   f"NDCG@20={r['ndcg@20']:.6f}\n")

    print(f"✓ Human-readable summary saved to: {summary_txt_file}")
else:
    print("\n⚠️ WARNING: No results were extracted from any training run!")
    print("Please check the log files manually.")


RESULTS SUMMARY (Mean ± Std across 5 runs)

RECALL@20:
  Mean: 0.091161
  Std:  0.000967
  Mean ± Std: 0.091161 ± 0.000967
  Individual values: ['0.091217', '0.092778', '0.090363', '0.090005', '0.091444']

PRECISION@20:
  Mean: 0.004830
  Std:  0.000045
  Mean ± Std: 0.004830 ± 0.000045
  Individual values: ['0.004839', '0.004896', '0.004785', '0.004775', '0.004852']

NDCG@20:
  Mean: 0.041765
  Std:  0.000214
  Mean ± Std: 0.041765 ± 0.000214
  Individual values: ['0.042055', '0.041953', '0.041527', '0.041536', '0.041752']

COMPARISON WITH REFERENCE BASELINE (BM3 - Amazon Baby)
NOTE: MMHCL paper does not report Amazon Baby results.
      Reference: BM3 (WWW '23) Table 3 - per-user 8:1:1 split on Baby.
  RECALL@20: Ours=0.0912 vs Paper=0.0883 (+3.2%) [✓ OK]
  PRECISION@20: Ours=0.0048 vs Paper=0.0044 (+9.8%) [✓ OK]
  NDCG@20: Ours=0.0418 vs Paper=0.0383 (+9.0%) [✓ OK]

✓ Summary saved to: ../Baby/multi_seed_summary.json
✓ Human-readable summary saved to: ../Baby/multi_seed_summary.txt

## 5. Export Results to Excel

In [5]:
from __future__ import annotations

from typing import Any

import pandas as pd
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

if len(all_results) > 0:
    print("="*80)
    print("EXPORTING RESULTS TO EXCEL")
    print("="*80)

    excel_file: str = f"../{dataset}/MMHCL_{dataset}_Results_Local.xlsx"

    # Prepare Individual Results data
    individual_data: list[dict[str, Any]] = []
    for i, r in enumerate(all_results, 1):
        row: dict[str, Any] = {
            'Run': i,
            'Seed': r['seed'],
            'Recall@20': r['recall@20'],
            'Precision@20': r['precision@20'],
            'NDCG@20': r['ndcg@20']
        }
        individual_data.append(row)

    df_individual: pd.DataFrame = pd.DataFrame(individual_data)

    # Prepare Summary Statistics data
    summary_data_list: list[dict[str, Any]] = []
    for metric in metrics:
        metric_name: str = metric.upper()
        summary_data_list.append({
            'Metric': metric_name,
            'Mean': stats[metric]['mean'],
            'Std': stats[metric]['std'],
            'Mean ± Std': f"{stats[metric]['mean']:.6f} ± {stats[metric]['std']:.6f}"
        })

    df_summary: pd.DataFrame = pd.DataFrame(summary_data_list)

    # Paper comparison data
    comparison_data: list[dict[str, Any]] = []
    for metric in metrics:
        our_mean: Any = stats[metric]['mean']
        paper_val: float = paper_values[metric]
        diff_pct: float = ((our_mean - paper_val) / paper_val) * 100
        comparison_data.append({
            'Metric': metric.upper(),
            'Our Result': our_mean,
            'Paper Result': paper_val,
            'Difference (%)': diff_pct
        })

    df_comparison: pd.DataFrame = pd.DataFrame(comparison_data)

    # Write to Excel with formatting
    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        df_individual.to_excel(writer, sheet_name='Individual Results', index=False)
        df_summary.to_excel(writer, sheet_name='Summary Statistics', index=False)
        df_comparison.to_excel(writer, sheet_name='Paper Comparison', index=False)

        # Format sheets
        for sheet_name in writer.sheets:
            ws = writer.sheets[sheet_name]
            # Auto-adjust column widths
            for col in ws.columns:
                max_length: int = 0
                column: str = col[0].column_letter
                for cell in col:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                ws.column_dimensions[column].width = max_length + 2

            # Bold header row
            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='E0E0E0', end_color='E0E0E0', fill_type='solid')

    print(f"\n✓ Results exported to: {excel_file}")
    print(f"  - Sheet 1: Individual Results ({len(all_results)} runs)")
    print(f"  - Sheet 2: Summary Statistics")
    print(f"  - Sheet 3: Paper Comparison")
else:
    print("No results to export.")

EXPORTING RESULTS TO EXCEL

✓ Results exported to: ../Baby/MMHCL_Baby_Results_Local.xlsx
  - Sheet 1: Individual Results (5 runs)
  - Sheet 2: Summary Statistics
  - Sheet 3: Paper Comparison


## 6. Check W&B Run Status (Optional)

In [6]:
from __future__ import annotations

import wandb

print(f"Checking runs for {wandb_entity}/{wandb_project}...")

try:
    api: wandb.Api = wandb.Api()
    runs: wandb.apis.public.Runs = api.runs(f"{wandb_entity}/{wandb_project}")

    if len(runs) == 0:
        print("No runs found in this project yet.")
    else:
        print(f"\nFound {len(runs)} runs. Latest 5 status:")
        print("-" * 60)
        for run in runs[:5]:
            print(f"Run Name: {run.name}")
            print(f"Status:   {run.state}")
            print(f"Created:  {run.created_at}")
            if 'val/recall' in run.summary:
                print(f"Val Recall: {run.summary['val/recall']:.4f}")
            if 'best_recall' in run.summary:
                print(f"Best Recall: {run.summary['best_recall']:.4f}")
            print("-" * 60)

except Exception as e:
    print(f"Error fetching run status: {e}")
    print("Make sure W&B is logged in correctly.")

Checking runs for baitapck51cc-uet/snips-local-mmhcl-baby...

Found 21 runs. Latest 5 status:
------------------------------------------------------------
Run Name: seed_1258187153
Status:   crashed
Created:  2026-03-07T12:12:26Z
------------------------------------------------------------
Run Name: seed_1302020602
Status:   finished
Created:  2026-03-07T12:41:29Z
Best Recall: 0.0768
------------------------------------------------------------
Run Name: seed_186809443
Status:   finished
Created:  2026-03-07T13:33:18Z
Best Recall: 0.0758
------------------------------------------------------------
Run Name: seed_1450008175
Status:   finished
Created:  2026-03-07T14:31:56Z
Best Recall: 0.0757
------------------------------------------------------------
Run Name: seed_628859525
Status:   finished
Created:  2026-03-08T11:29:11Z
Best Recall: 0.0840
------------------------------------------------------------


## Troubleshooting

### Common Issues:

1. **CUDA Out of Memory**: Reduce batch size further (256 or 128)
   ```python
   batch_size = 256  # or 128
   ```

2. **W&B login issues**: Run `wandb login` in terminal first

3. **Module not found**: Install dependencies
   ```bash
   pip install torch wandb numpy pandas openpyxl scipy
   ```

4. **Data not found**: Ensure all preprocessed files are present in `data/Baby/`:
   - `data/Baby/5-core/train.json`
   - `data/Baby/5-core/val.json`
   - `data/Baby/5-core/test.json`
   - `data/Baby/image_feat.npy`  (CLIP ViT-B/32, 512-d, 7050 items)
   - `data/Baby/text_feat.npy`   (CLIP ViT-B/32, 512-d, 7050 items)

5. **KeyError / shape mismatch on image/text features**: Re-run
   `reextract_image_features.py` and `verify_and_fix_text_features.py` from project root.

### Reference Results (Amazon Baby):
The original MMHCL paper does **not** report results on Amazon Baby.
BM3 (WWW '23) Table 3 reference baseline (per-user 8:1:1 random split):
- Recall@20:    0.0883
- Precision@20: 0.0044
- NDCG@20:      0.0383

Both MMHCL and BM3 use per-user 8:1:1 random split, ensuring all 19,445 users are evaluated.